# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and process record sets from the FAIR^2 Mass Spectrometry EV dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library and Python.

### Dataset Source
The dataset is described using the Croissant schema standard. All entities are referenced via their unique `@id` field, as specified by the FAIR^2 package. You can access the schema here:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading

We start by loading the dataset Croissant description and metadata using `mlcroissant`. Afterward, we'll display a summary of the dataset including the title and main description.

In [ ]:
import mlcroissant as mlc
import pandas as pd

croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset via Croissant
dataset = mlc.Dataset(croissant_url)

# dataset.metadata is an object, access via attributes
print(f"Title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"License: {dataset.metadata.license}")
print(f"Keywords: {', '.join(dataset.metadata.keywords)}")

## 2. Data Overview

We will now explore the dataset's available record sets, and for each, list their available fields (with their `@id`s) for structured exploration. All identifiers are always referenced by their unique `@id`.

In [ ]:
# Helper functions to list record sets and their fields

def print_record_sets(ds):
    print("Available record sets (@id and name):")
    for rs in ds.metadata.record_sets:
        print(f"- @id: {rs['@id']}, name: {rs['name']}")

def print_fields_of_record_set(ds, record_set_id):
    record_set = None
    for rs in ds.metadata.record_sets:
        if rs['@id'] == record_set_id:
            record_set = rs
            break
    if record_set:
        print(f"Fields for Record Set '{record_set['name']}' (@id: {record_set_id}):")
        for field in record_set.get('fields', []):
            print(f"  - @id: {field['@id']}, name: {field['name']}, dataType: {field.get('dataType', None)}")
    else:
        print(f"No record set found with @id {record_set_id}")

# List record sets
print_record_sets(dataset)

# As an example, print fields in the main record set (you may list all sets and pick by name/@id)
record_sets = [rs['@id'] for rs in dataset.metadata.record_sets]
if record_sets:
    first_record_set_id = record_sets[0]
    print_fields_of_record_set(dataset, first_record_set_id)

## 3. Data Extraction

Let's extract data from the record sets above. For every record set, we extract records via its `@id`, and assemble the results as a Pandas DataFrame.

> Note: If you wish to work with a specific record set, replace the `record_sets` list with the desired `@id`s. All further analysis will reference record/field/column `@id`s.

In [ ]:
# Extract data from each record set
record_sets = [rs['@id'] for rs in dataset.metadata.record_sets]  # All record sets by @id
dataframes = {}

for record_set_id in record_sets:
    print(f"Reading records from: {record_set_id}")
    try:
        records_iter = dataset.records(record_set=record_set_id)
        df = pd.DataFrame(records_iter)
        dataframes[record_set_id] = df
        print(f"  -> {df.shape[0]} records, columns: {list(df.columns)}")
    except Exception as e:
        print(f"  Error loading record set {record_set_id}: {e}")

# Example: Show first few columns and rows for the main set
if record_sets:
    main_record_set_id = record_sets[0]
    print(f"\nColumns for {main_record_set_id}: {list(dataframes[main_record_set_id].columns)}")
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Some typical EDA steps: filtering numeric fields, normalizing values, or grouping by categorical attributes using only `@id` references.

* Replace variable values below with IDs from the previous cell's DataFrame columns for your analysis.

In [ ]:
# Example IDs; these should correspond to the `@id` from field definition (see markdown above and DataFrame columns)
record_set_id = main_record_set_id  # Use the main record set
df = dataframes[record_set_id]

# Example field IDs (replace as required based on previous output)
# Let's assume the field @id for molecular weight is 'mw' and for protein name is 'description'.
numeric_field_id = None
group_field_id = None
# Try to auto-detect fields commonly present in protein tables
for col in df.columns:
    # Try common molecular weight column names
    if col.lower() in ['mw', '@mw', 'molecular_weight', 'cr:mw']:
        numeric_field_id = col
    elif col.lower() in ['description', '@description', 'protein_name', 'cr:description']:
        group_field_id = col

# Fallback if not detected
if not numeric_field_id:
    numeric_field_id = df.columns[0]  # select the first numeric column

print(f"Selected numeric field: {numeric_field_id}")
if group_field_id:
    print(f"Selected group (categorical) field: {group_field_id}")

# Drop NA and convert to numeric for EDA
df_eda = df.copy()
df_eda[numeric_field_id] = pd.to_numeric(df_eda[numeric_field_id], errors='coerce')
df_eda = df_eda.dropna(subset=[numeric_field_id])

# Filtering: Show records with high values
if not df_eda.empty:
    threshold = df_eda[numeric_field_id].quantile(0.9)
    filtered = df_eda[df_eda[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered.head())

    # Normalizing the numeric field
    norm_col = f"{numeric_field_id}_normalized"
    df_eda[norm_col] = (df_eda[numeric_field_id] - df_eda[numeric_field_id].mean()) / df_eda[numeric_field_id].std()
    print(f"First 5 normalized values for {numeric_field_id}:")
    display(df_eda[[numeric_field_id, norm_col]].head())

    if group_field_id and group_field_id in df_eda.columns:
        grouped_stats = df_eda.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped means by {group_field_id}:")
        display(grouped_stats.head())

## 5. Visualization

Let's visualize distribution of a numeric field and, if available, its distribution grouped by another attribute.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 4))
sns.histplot(df_eda[numeric_field_id], bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

# If group field exists, plot the top categories
if group_field_id and group_field_id in df_eda.columns:
    plt.figure(figsize=(10, 6))
    # Show only top 10 categories
    top_cats = df_eda[group_field_id].value_counts().index[:10]
    sns.boxplot(
        x=group_field_id,
        y=numeric_field_id,
        data=df_eda[df_eda[group_field_id].isin(top_cats)]
    )
    plt.title(f"{numeric_field_id} distribution by {group_field_id} (top 10)")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

This notebook illustrated how to:
- Load dataset metadata and records described by a Croissant schema,
- Explore record sets and fields via their `@id` values,
- Extract structured records into DataFrames,
- Apply basic filtering, normalization, and grouping for EDA,
- Visualize distributions for key attributes.

All processing references original field `@id` to ensure reproducibility and clarity across dataset versions. For complete details on available attributes, reference the [Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) directly, or explore further using `mlcroissant` metadata methods.